# Bibliotecas

In [1]:
import numpy as np
import os
import pandas as pd
from datetime import datetime, timedelta
import random

# Parametros

In [2]:
PRODUCT_NAMES = ['suculenta', 'cacto', 'girasol']
NUM_PRODUTO = len(PRODUCT_NAMES)
NUM_AMBIENTE = 50
NUM_CATEGORIA = 1
NUM_PREVISAO = 200
NUM_TEMPO = 365
NUM_FACT = 1000

LOTE_CARGA = 1
USUARIOS = [f'user_{i}' for i in range(1, 6)]
POSICOES = ['Prateleira A', 'Prateleira B', 'Corredor 1', 'Corredor 2', 'Armazem']
TIPOS_PREVISAO = ['venda', 'qualidade']
SOBREVIVENCIA_STATES = ['Viva', 'Morta', 'Vendida', 'Descartada']

# 1. Dimension: Ambiente

In [3]:
ambiente = pd.DataFrame({
    'sk_ambiente': range(1, NUM_AMBIENTE + 1),
    'nr_temperatura': np.round(np.random.uniform(10, 35, NUM_AMBIENTE), 2),
    'nr_umidade': np.round(np.random.uniform(20, 80, NUM_AMBIENTE), 2),
    'ds_posicao': np.random.choice(POSICOES, NUM_AMBIENTE),
    'nr_usuario': np.random.choice(USUARIOS, NUM_AMBIENTE),
    'nr_lote_carga': LOTE_CARGA
})

# 2. Dimension: Categoria

In [4]:
categoria = pd.DataFrame({
    'sk_categoria': NUM_CATEGORIA ,
    'nm_categoria': 'Jardinagem',
    'ds_categoria': 'A jardinagem é a prática de cultivo, manutenção e design de jardins, abrangendo plantas ornamentais, hortas e paisagismo. Inclui atividades como plantio, irrigação, poda, controle de pragas, adubação e criação de espaços como jardins verticais, de inverno, etc.',
    'nr_usuario': np.random.choice(USUARIOS, NUM_CATEGORIA),
    'nr_lote_carga': LOTE_CARGA
})

# 3. Dimension: Produto

In [5]:
produto = pd.DataFrame({
    'sk_produto': range(1, NUM_PRODUTO + 1),
    'nm_produto': PRODUCT_NAMES,
    'ds_produto': ' ',
    'nr_val': np.round(np.random.uniform(5, 500, NUM_PRODUTO), 2),
    'nr_val_saida': np.round(np.random.uniform(5, 500, NUM_PRODUTO), 2),
    'nr_usuario': np.random.choice(USUARIOS, NUM_PRODUTO),
    'nr_lote_carga': LOTE_CARGA
})

# 4. Dimension: Previsao

In [6]:
start_date = datetime.today() - timedelta(days=365)
dates = [start_date + timedelta(days=i) for i in range(NUM_PREVISAO)]
previsao = pd.DataFrame({
    'sk_previsao': range(1, NUM_PREVISAO + 1),
    'dt_previsao': np.random.choice(dates, NUM_PREVISAO),
    'nr_acuracia': np.round(np.random.uniform(50, 100, NUM_PREVISAO), 2),
    'tp_previsao': np.random.choice(TIPOS_PREVISAO, NUM_PREVISAO),
    'nr_usuario': np.random.choice(USUARIOS, NUM_PREVISAO),
    'nr_lote_carga': LOTE_CARGA
})

# 5. Dimension: Tempo

In [7]:
base_date = datetime.today() - timedelta(days=NUM_TEMPO)
tempo = []
for i in range(NUM_TEMPO):
    dt = base_date + timedelta(days=i)
    tempo.append({
        'sk_tempo': i + 1,
        'dt_evento': dt,
        'nr_hora': np.random.randint(0, 24),
        'nr_dia': dt.day,
        'nr_mes': dt.month,
        'nr_ano': dt.year,
        'nr_usuario': random.choice(USUARIOS),
        'nr_lote_carga': LOTE_CARGA
    })
tempo = pd.DataFrame(tempo)

# 6. Fact Table: Estoque

In [8]:
fact = pd.DataFrame({
    'sk_estoque': range(1, NUM_FACT + 1),
    'sk_categoria': np.random.choice(categoria['sk_categoria'], NUM_FACT),
    'sk_ambiente': np.random.choice(ambiente['sk_ambiente'], NUM_FACT),
    'sk_previsao': np.random.choice(previsao['sk_previsao'], NUM_FACT),
    'sk_tempo': np.random.choice(tempo['sk_tempo'], NUM_FACT),
    'sk_produto': np.random.choice(produto['sk_produto'], NUM_FACT),
    'st_sobrevivencia': np.random.choice(SOBREVIVENCIA_STATES, NUM_FACT),
    'nr_media_previsao': np.round(np.random.uniform(0, 100, NUM_FACT), 2),
    'nr_media_qualidade': np.round(np.random.uniform(0, 100, NUM_FACT), 2),
    'nr_qtd': np.random.randint(1, 100, NUM_FACT),
    'nr_lucro': np.round(np.random.uniform(-50, 200, NUM_FACT), 2),
    'nr_prejuizo': np.round(np.random.uniform(0, 50, NUM_FACT), 2)
})

# 7. Denormalize: Juntando as tabelas dimenção com a fato

In [9]:
output_folder = os.path.join(os.getcwd(), 'dados\datawarehouse')
os.makedirs(output_folder, exist_ok=True)

tables = {
    'dim_ambiente':   ambiente,
    'dim_categoria':  categoria,
    'dim_produto':    produto,
    'dim_previsao':   previsao,
    'dim_tempo':      tempo,
    'fato_base':      fact
}

# 3) Loop que salva tudo
for table_name, df in tables.items():
    file_path = os.path.join(output_folder, f"{table_name}.csv")
    df.to_csv(file_path, index=False)
    print(f"Exported {table_name} → {file_path}")

Exported dim_ambiente → c:\Users\di\Desktop\sprint4\Sprint_4\Mirror-Digitale\Geradores de dados\dados\datawarehouse\dim_ambiente.csv
Exported dim_categoria → c:\Users\di\Desktop\sprint4\Sprint_4\Mirror-Digitale\Geradores de dados\dados\datawarehouse\dim_categoria.csv
Exported dim_produto → c:\Users\di\Desktop\sprint4\Sprint_4\Mirror-Digitale\Geradores de dados\dados\datawarehouse\dim_produto.csv
Exported dim_previsao → c:\Users\di\Desktop\sprint4\Sprint_4\Mirror-Digitale\Geradores de dados\dados\datawarehouse\dim_previsao.csv
Exported dim_tempo → c:\Users\di\Desktop\sprint4\Sprint_4\Mirror-Digitale\Geradores de dados\dados\datawarehouse\dim_tempo.csv
Exported fato_base → c:\Users\di\Desktop\sprint4\Sprint_4\Mirror-Digitale\Geradores de dados\dados\datawarehouse\fato_base.csv


In [10]:
denorm = (
    fact
    .merge(produto, on='sk_produto')
    .merge(categoria, on='sk_categoria')
    .merge(ambiente, on='sk_ambiente')
    .merge(previsao, on='sk_previsao', suffixes=('_produto', '_previsao'))
    .merge(tempo, on='sk_tempo', suffixes=('_previsao', '_tempo'))
)

# 8. Exportando os dados em um unico CSV

In [11]:
output_folder = 'dados'
os.makedirs(output_folder, exist_ok=True)
output_filename = os.path.join(output_folder, 'ESTOQUE_SIMULADO.csv')

denorm.to_csv(output_filename, index=False)

print(f"Denormalized fact CSV generated: {output_filename}")

Denormalized fact CSV generated: dados\ESTOQUE_SIMULADO.csv
